# PharmaFlow Foundation — Session 1.2 Solutions
### Functions, Modules & File I/O

These are the instructor solutions for all five exercise tiers. Every cell below has
been run against the real dataset for this session (`adverse_events_100.json`) — the
outputs shown are actual output, not illustrative text.

**Dataset:** the same 100 real FDA adverse event reports from Session 1.1
(`adverse_events_100.json`), plus a new reference file this session adds,
`country_codes.csv` — 54 rows mapping each country code (including ones outside this
100-record file, to also cover the 1,000-record Lab pull) to
its full country name and business region (North America / EMEA / APAC / LATAM).


## Before You Run Anything — Environment Check

Same setup pattern as Session 1.1 — this notebook finds the repo automatically,
no matter which folder your notebook tool happened to start in.

Run the cell below first. If it prints an error instead of "You're ready to
go.", stop here and fix your setup before continuing — every exercise from
this point on depends on the `DATA_FILE` and `LOOKUP_FILE` variables it defines.


In [1]:
# --- Setup check: always run this cell first ---
import os

def find_repo_root(marker_folder="data", start=None):
    """
    Same reasoning as Session 1.1: different notebook tools start a
    notebook's kernel in different folders, so we search upward from
    wherever we happened to start until we find a folder that contains
    `data` -- that only exists at the repo root, so wherever we find it
    IS the repo root.
    """
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, marker_folder)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(
                f"Could not find a '{marker_folder}' folder anywhere above "
                f"{os.getcwd()}. Make sure this notebook is somewhere inside "
                f"the pharmaflow-ai repo."
            )
        current = parent


REPO_ROOT = find_repo_root()
DATA_FILE = os.path.join(REPO_ROOT, "data", "raw", "adverse_events_100.json")
LOOKUP_FILE = os.path.join(REPO_ROOT, "data", "raw", "country_codes.csv")
OUTPUT_FILE = os.path.join(REPO_ROOT, "data", "processed", "adverse_events_cleaned.json")

missing = [p for p in (DATA_FILE, LOOKUP_FILE) if not os.path.exists(p)]
if not missing:
    print("Repo root found at:", REPO_ROOT)
    print("Data file found at:", DATA_FILE)
    print("Lookup file found at:", LOOKUP_FILE)
    print("You're ready to go.")
else:
    print("Repo root found at:", REPO_ROOT)
    print("Missing file(s):", missing)
    print()
    print("Fix: confirm both adverse_events_100.json and country_codes.csv are in")
    print("data/raw/ at the top level of the pharmaflow-ai repo.")


Repo root found at: /home/claude/work/final
Data file found at: /home/claude/work/final/data/raw/adverse_events_100.json
Lookup file found at: /home/claude/work/final/data/raw/country_codes.csv
You're ready to go.


## Warm-Up — Save it, then trust it (no FDA data yet)

Before writing any function that reads or writes real FDA data, prove to
yourself that a write-then-read-back round-trip actually reproduces the same
data, on 6 invented records small enough to check by eye.


In [1]:
# WARM-UP: tiny invented data, no FDA records yet.
# Same toy_scores as Session 1.1.
import json

toy_scores = [
    {"student": "Asha", "subject": "Math", "score": 88},
    {"student": "Asha", "subject": "Science", "score": 73},
    {"student": "Ravi", "subject": "Math", "score": 65},
    {"student": "Ravi", "subject": "Science", "score": 91},
    {"student": "Meera", "subject": "Math", "score": 95},
    {"student": "Meera", "subject": "Science", "score": 95},
]

def average_score(records):
    """Return the average 'score' value across a list of record dicts."""
    return sum(r["score"] for r in records) / len(records)

print(f"Average toy score: {average_score(toy_scores)}")

# Write it out, then read it back -- prove the round-trip is exact.
TOY_FILE = os.path.join(REPO_ROOT, "data", "processed", "toy_scores.json")
os.makedirs(os.path.dirname(TOY_FILE), exist_ok=True)

with open(TOY_FILE, "w") as f:
    json.dump(toy_scores, f, indent=2)

with open(TOY_FILE) as f:
    reloaded = json.load(f)

print(f"Round-trip match: {reloaded == toy_scores}")


Average toy score: 84.5
Round-trip match: True


## Exercise 1 (Beginner) — One function, one job: `load_data()`


In [1]:
# Tier 1 - Beginner: a loading function that does exactly one thing.

def load_data(filepath):
    """Load a JSON file of adverse event records and return them as a list
    of dicts. Does exactly one thing: read the file, parse it, return it --
    no cleaning, no filtering, no printing. That belongs to other functions,
    so this one stays reusable by anything that just needs the raw records.
    """
    with open(filepath) as f:
        return json.load(f)

records = load_data(DATA_FILE)
print(f"Loaded {len(records)} records")
print("First record:", records[0])


Loaded 100 records
First record: {'safety_report_id': '10003314', 'drug_name': 'LETAIRIS', 'reaction': 'Back pain', 'serious': '2', 'patient_age': '82', 'country': 'US', 'event_date': '20140312'}


## Exercise 2 (Intermediate) — Clean it, then save it: `clean_records()` + `write_output()`

Reuses `load_data()` from Exercise 1 — run that cell first if you haven't.


In [1]:
# Tier 2 - Intermediate: two separate responsibilities -- clean in memory,
# then persist to disk -- kept as two functions on purpose.

records = load_data(DATA_FILE)  # reload fresh, same as Exercise 1

def clean_records(records):
    """Return a NEW list of records with patient_age converted from string
    to int where present, and an explicit age_known flag added. Does not
    modify the input list or its dicts -- a caller's original data should
    never change just because they passed it to this function.
    """
    cleaned = []
    for r in records:
        new_r = dict(r)
        age_raw = new_r.get("patient_age")
        if age_raw is not None:
            new_r["patient_age"] = int(age_raw)
            new_r["age_known"] = True
        else:
            new_r["age_known"] = False
        cleaned.append(new_r)
    return cleaned

def write_output(records, filepath):
    """Write a list of records to a JSON file, pretty-printed. Assumes the
    records are already clean -- this function's only job is saving them.
    """
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w") as f:
        json.dump(records, f, indent=2)

cleaned = clean_records(records)
known = sum(1 for r in cleaned if r["age_known"])
print(f"{known} of {len(cleaned)} records have a known patient_age")
print("Sample cleaned record:", cleaned[0])

write_output(cleaned, OUTPUT_FILE)
print(f"Wrote {len(cleaned)} cleaned records to {OUTPUT_FILE}")

# Pick a record known to HAVE patient_age (not just records[0] -- not every
# record has this field, so indexing [0] would break on a differently-ordered
# or differently-curated file even though it happens to work on this one).
sample_with_age = next(r for r in records if "patient_age" in r)
print("A record's patient_age is still a string (not mutated):",
      type(sample_with_age["patient_age"]).__name__)


69 of 100 records have a known patient_age
Sample cleaned record: {'safety_report_id': '10003314', 'drug_name': 'LETAIRIS', 'reaction': 'Back pain', 'serious': '2', 'patient_age': 82, 'country': 'US', 'event_date': '20140312', 'age_known': True}
Wrote 100 cleaned records to /home/claude/work/final/data/processed/adverse_events_cleaned.json
A record's patient_age is still a string (not mutated): str


**Trainer note for this output:** 69 of 100 — not all 100 — have a known
`patient_age`, the same 31-missing split from Session 1.1. That's expected:
real submitters don't always report every field, and `clean_records()`'s job
is to make that gap explicit (`age_known: False`), not to hide or fabricate it.


## Exercise 3 (Advanced) — Names, not codes: reading a second file with `csv.DictReader`

Reuses `cleaned` from Exercise 2 — run that cell first if you haven't.


In [1]:
# Tier 3 - Advanced: a second file this time -- a CSV reference table --
# read with csv.DictReader and joined onto the records by country code.
import csv
from collections import Counter

def load_country_lookup(filepath):
    """Read the country-code reference CSV and return a dict mapping
    country_code -> {"country_name": ..., "business_region": ...}.
    """
    lookup = {}
    with open(filepath, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            lookup[row["country_code"]] = {
                "country_name": row["country_name"],
                "business_region": row["business_region"],
            }
    return lookup

def enrich_with_country_info(records, lookup):
    """Return a NEW list of records with country_name and business_region
    added, looked up from the reference CSV by country code. Never crashes
    on a code with no match -- flags it instead.
    """
    enriched = []
    missing = set()
    for r in records:
        new_r = dict(r)
        code = new_r.get("country")
        info = lookup.get(code)
        if info is None:
            missing.add(code)
            new_r["country_name"] = "UNKNOWN"
            new_r["business_region"] = "UNKNOWN"
        else:
            new_r["country_name"] = info["country_name"]
            new_r["business_region"] = info["business_region"]
        enriched.append(new_r)
    if missing:
        print(f"WARNING: {len(missing)} country code(s) had no lookup match: {missing}")
    else:
        print("Every record's country code matched the lookup -- no gaps.")
    return enriched

country_lookup = load_country_lookup(LOOKUP_FILE)
print(f"Loaded {len(country_lookup)} country lookup entries")

enriched = enrich_with_country_info(cleaned, country_lookup)
print("Sample enriched record:", enriched[0])

region_counts = Counter(r["business_region"] for r in enriched)
for region, count in region_counts.most_common():
    print(f"{region:<15} {count}")


Loaded 54 country lookup entries
Every record's country code matched the lookup -- no gaps.
Sample enriched record: {'safety_report_id': '10003314', 'drug_name': 'LETAIRIS', 'reaction': 'Back pain', 'serious': '2', 'patient_age': 82, 'country': 'US', 'event_date': '20140312', 'age_known': True, 'country_name': 'United States', 'business_region': 'North America'}
North America   72
EMEA            19
APAC            6
LATAM           3


## Exercise 4 (Corporate Scenario) — The 2am crash, refactored for good


In [1]:
# Tier 4 - Corporate Scenario: this is the exact load_data() / clean_records() /
# write_output() split from Exercises 1-2, but this time proving WHY it matters --
# no hardcoded paths (find_repo_root() instead), no bare open() left unclosed
# (a `with` block for every file), and pathlib for building the output path.
from pathlib import Path

def find_repo_root_pathlib(marker_folder="data", start=None):
    """Same idea as find_repo_root() above, written with pathlib instead of
    os.path -- both styles are worth knowing, you'll see both in real code."""
    current = Path(start or os.getcwd()).resolve()
    while True:
        if (current / marker_folder).is_dir():
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find a '{marker_folder}' folder anywhere above {os.getcwd()}."
            )
        current = current.parent

ROOT2 = find_repo_root_pathlib()
DATA_FILE2 = ROOT2 / "data" / "raw" / "adverse_events_100.json"
OUTPUT_FILE2 = ROOT2 / "data" / "processed" / "adverse_events_cleaned.json"

def write_output_pathlib(records, filepath):
    """Write a list of records to a JSON file, pretty-printed -- pathlib
    version of write_output(), same job, Path object instead of a string."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w") as f:
        json.dump(records, f, indent=2)

records2 = load_data(DATA_FILE2)           # load_data() doesn't care whether
cleaned2 = clean_records(records2)         # filepath is a str or a Path
write_output_pathlib(cleaned2, OUTPUT_FILE2)

print(f"Wrote {len(cleaned2)} records to {OUTPUT_FILE2}")

# Same fix as Exercise 2: pick a record known to HAVE patient_age rather than
# assuming records2[0] is one of them.
sample2_with_age = next(r for r in records2 if "patient_age" in r)
print("Original (records2) still has string patient_age -- proves clean_records()")
print("didn't mutate its input, the same guarantee as Exercise 2:",
      type(sample2_with_age["patient_age"]).__name__)


Wrote 100 records to /home/claude/work/final/data/processed/adverse_events_cleaned.json
Original (records2) still has string patient_age -- proves clean_records()
didn't mutate its input, the same guarantee as Exercise 2: str


## Exercise 5 (Interview Challenge) — One function, no assumptions


In [1]:
# Tier 5 - Interview Challenge: framed exactly as it would come up live --
# "write a function that counts records in a JSON file, and it must never
# crash, no matter what path you hand it."

def count_valid_records(filepath):
    """Return the number of records in a JSON file at filepath, or 0 if the
    file is missing, isn't valid JSON, or doesn't contain a list -- never
    raises, whatever path it's given.
    """
    try:
        with open(filepath) as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return 0
    if not isinstance(data, list):
        return 0
    return len(data)

print("Real file:", count_valid_records(DATA_FILE))
print("Missing file:", count_valid_records(os.path.join(REPO_ROOT, "data", "raw", "nope.json")))

BAD_JSON = os.path.join(REPO_ROOT, "data", "raw", "_bad_for_test.json")
with open(BAD_JSON, "w") as f:
    f.write("{not valid json")
print("Malformed JSON:", count_valid_records(BAD_JSON))

DICT_JSON = os.path.join(REPO_ROOT, "data", "raw", "_dict_for_test.json")
with open(DICT_JSON, "w") as f:
    json.dump({"not": "a list"}, f)
print("Valid JSON, but a dict not a list:", count_valid_records(DICT_JSON))

# clean up the two throwaway test files
os.remove(BAD_JSON)
os.remove(DICT_JSON)


Real file: 100
Missing file: 0
Malformed JSON: 0
Valid JSON, but a dict not a list: 0


---
### Grading notes

- Exercise 2's 69/31 known/unknown `patient_age` split matches Session 1.1's real-data
  finding exactly -- same file, same gap. If a student's number differs, check
  whether they're converting `patient_age` before or after checking for `None`.
- Exercise 3's region counts (North America 72 / EMEA 19 / APAC 6 / LATAM 3) depend
  on `country_codes.csv` having a matching row for all 17 codes that actually appear
  in this 100-record file (the file itself has 54 rows total, since it also covers
  the wider 1,000-record Lab pull) --
  if a student's lookup is missing an entry, they'll see the "no lookup match"
  warning fire, which is the intended defensive behavior, not a bug to silence.
- Exercise 4 is intentionally the same three functions as Exercises 1-2 -- grade
  for the refactor discipline (find_repo_root, `with`, pathlib, no mutation), not
  for novel logic; a student who gets here having genuinely internalized
  Exercises 1-2 should find this one fast.
- Exercise 5 must return `0`, not raise, for all three failure cases (missing file,
  malformed JSON, wrong type) -- a student who lets even one of these raise hasn't
  fully solved it, even if the happy path works.
